In [1]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import scipy
from itertools import combinations

In [2]:
sys.path.insert(0, "src")

from grn import grn

np.random.seed(5)


In [3]:
#manually adjust GRN hyperparameters in an attempt to build GRN with more non-additive effects
G = grn().add_structure(
    n=256,
    k=1,
    alpha=1e-99,
    beta=0.99,       # 1 - 1/r
    gamma=0.01,      # 1/r
    kappa=10,        # --w
    delta_in=100,
    delta_out=100,
)

In [4]:
G.simulate_rna(
    s=1e-4,
    dt=1e-2,
    tmax=20000,
    burnin=5000,
    tol=1e-3,
    save=True,
)
x_wt = np.zeros(256)
y_wt = G.rna

In [5]:
single_ko_expression = np.zeros((G.n, G.n))
single_ko_logfc = np.zeros((G.n, G.n))
single_ko_convergence = {}

for gene in tqdm(range(G.n), desc="Simulating single-gene KOs"):
    
    result = G.ko_nodes(
        genes=[gene],
        stats=("new_rna", 'logfc', 'convergence'),
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )
    
    single_ko_expression[gene, :] = result["new_rna"].flatten()
    single_ko_logfc[gene, :] = result["logfc"].flatten()
    single_ko_convergence[gene] = result['convergence']
x_single_ko = np.eye(G.n)
y_single_ko = single_ko_expression

Simulating single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]

In [6]:
pairs = list(combinations(range(G.n), 2))

n_jobs = 64

def run_double_ko(pair):
    a, b = pair

    result = G.ko_nodes(
        genes=[a, b],
        stats=("new_rna", "logfc", 'convergence'),
        s=0,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )

    return (
        np.asarray(result["new_rna"]).flatten(),
        np.asarray(result["logfc"]).flatten(),
        np.asarray(result['convergence'])
    )


results = Parallel(
    n_jobs=64,
    pre_dispatch="n_jobs",
    prefer="processes",
    verbose=10,
)(
    delayed(run_double_ko)(pair)
    for pair in pairs
)

double_expression = np.asarray(
    [r[0] for r in results],
    dtype=np.float32
)

double_logfc = np.asarray(
    [r[1] for r in results],
    dtype=np.float32
)
double_convergence = np.asarray(
    [r[2] for r in results],
    dtype=np.float32
)



#now prepare as training data

pairs = np.array(
    pairs,
    dtype=np.int32
)

x_double_ko = np.zeros((pairs.shape[0], G.n))

#now fill x for doubleKO with 1s

rows = np.arange(pairs.shape[0])
x_double_ko[rows, pairs[:, 0]] = 1.0
x_double_ko[rows, pairs[:, 1]] = 1.0

y_double_ko = double_expression

[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done   8 tasks      | elapsed:   23.0s
[Parallel(n_jobs=64)]: Done  21 tasks      | elapsed:   23.3s
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:   23.5s
[Parallel(n_jobs=64)]: Done  49 tasks      | elapsed:   23.7s
[Parallel(n_jobs=64)]: Done  64 tasks      | elapsed:   23.9s
[Parallel(n_jobs=64)]: Done  81 tasks      | elapsed:   24.1s
[Parallel(n_jobs=64)]: Done  98 tasks      | elapsed:   24.5s
[Parallel(n_jobs=64)]: Done 117 tasks      | elapsed:   25.1s
[Parallel(n_jobs=64)]: Done 136 tasks      | elapsed:   25.8s
[Parallel(n_jobs=64)]: Done 157 tasks      | elapsed:   26.4s
[Parallel(n_jobs=64)]: Done 178 tasks      | elapsed:   27.0s
[Parallel(n_jobs=64)]: Done 201 tasks      | elapsed:   27.6s
[Parallel(n_jobs=64)]: Done 224 tasks      | elapsed:   28.1s
[Parallel(n_jobs=64)]: Done 249 tasks      | elapsed:   28.9s
[Parallel(n_jobs=64)]: Done 274 tasks      | elapsed:  

# Lets train MLP on toy 256 gene GRN

In [7]:
seed = 0

import torch
from torch import nn
torch.manual_seed(seed)
device = torch.device('cuda')
#for more precision
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [8]:
#model implementation without dropout
model = nn.Sequential(nn.Linear(256, 512), nn.GELU(), nn.Linear(512, 512), nn.GELU(), nn.Linear(512, 256)).to(device)
linear = nn.Linear(256, 256).to(device)

#implement AdamW
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
#process data, standardise, put them on GPU

rng = np.random.default_rng(42)

n_pairs = pairs.shape[0]
nt, nv = int(0.5 * n_pairs), int(0.1 * n_pairs)

indices = rng.permutation(n_pairs)
test_idx = indices[:nt]
val_idx = indices[nt:nt + nv]
train_idx = indices[nt + nv:]

xtr = np.concatenate([x_wt[None, :], x_single_ko, x_double_ko[train_idx]], axis=0)
ytr = np.concatenate([y_wt[None, :], y_single_ko, y_double_ko[train_idx]], axis=0)

xv = x_double_ko[val_idx]
yv = y_double_ko[val_idx]

x_test = x_double_ko[test_idx]
y_test = y_double_ko[test_idx]

mean, scale = np.mean(ytr, axis=0), np.maximum(ytr.std(axis=0), 1e-6)

s_ytr = (ytr - mean) / scale

xtr_t   = torch.as_tensor(xtr, dtype=torch.float32, device=device)
s_ytr_t = torch.as_tensor(s_ytr, dtype=torch.float32, device=device)

xv_t    = torch.as_tensor(xv, dtype=torch.float32, device=device)
yv_t    = torch.as_tensor(yv, dtype=torch.float32, device=device)

x_test_t = torch.as_tensor(x_test, dtype=torch.float32, device=device)
y_test_t = torch.as_tensor(y_test, dtype=torch.float32, device=device)

mean_t  = torch.as_tensor(mean, dtype=torch.float32, device=device)
scale_t = torch.as_tensor(scale, dtype=torch.float32, device=device)



In [ ]:
import time
import copy

max_epochs = 300
batch_size = 256
lr = 1e-3
patience = 40
min_lr = 1e-5
lr_patience = 12

best, best_epoch, stale, lr_stale = np.inf, 0, 0, 0
history = []
started = time.perf_counter()
best_model_weights = None
for epoch in range(1, max_epochs +1):
    model.train()
    permutation = torch.randperm(xtr_t.shape[0], device=device)
    total = torch.zeros((), device=device)
    for start in range(0, xtr_t.shape[0], batch_size):
        ii = permutation[start: (start + batch_size)]
        optimizer.zero_grad(set_to_none=True)
        loss = torch.mean((model(xtr_t[ii]) - s_ytr_t[ii]) ** 2)
        loss.backward()
        optimizer.step()
        total += loss.detach() * len(ii)
    train_loss = float(total.item() / xtr_t.shape[0])
    model.eval()
    with torch.inference_mode():
        val_scaled = torch.cat([model(chunk) for chunk in xv_t.split(1024)]).cpu().numpy()
    val_pred = val_scaled.astype(np.float32) * scale + mean
    raw_mse = float(np.mean((val_pred - yv) ** 2))
    ####
    if not np.isfinite(train_loss) or not np.isfinite(raw_mse):
        raise FloatingPointError(f'Non-finite loss at epoch {epoch}')
    history.append({'epoch': epoch, 'train_scaled_mse': train_loss,
                        'val_scaled_mse': float(np.mean(((val_pred-yv)/scale)**2)),
                        'val_raw_mse': raw_mse, 'learning_rate': lr,
                        'elapsed_seconds': time.perf_counter()-started})
    if raw_mse < best:
        best, best_epoch, stale, lr_stale = raw_mse, epoch, 0, 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        stale += 1
        lr_stale += 1
    if epoch == 1 or epoch % 10 == 0:
        print(f'epoch {epoch}: train scaled MSE {train_loss:.6g}; '
                f'val RMSE {raw_mse**.5:.7g}; lr {lr:g}; '
                f'{time.perf_counter()-started:.0f}s', flush=True)
    if lr_stale >= lr_patience and lr > min_lr:
        lr = max(min_lr, lr * .5)
        for group in optimizer.param_groups:
            group['lr'] = lr
        lr_stale = 0
    if stale >= patience:
        break

if best_model_weights is not None:
    model.load_state_dict(best_model_weights)

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Initialize standard Linear Regression (no alpha/penalty)
linear_model = LinearRegression(fit_intercept=True)

# 2. Fit directly on the NumPy arrays
linear_model.fit(xtr.astype(np.float64), s_ytr.astype(np.float64))

# 3. Predict and reverse normalization
val_scaled = linear_model.predict(xv.astype(np.float64))
val_pred = val_scaled.astype(np.float32) * scale + mean

# 4. Calculate RMSE
raw_mse = float(np.mean((val_pred - yv) ** 2))
print(f"Standard Linear Baseline Validation RMSE: {raw_mse**.5:.7g}")

In [ ]:
# 1. Generate predictions for the Test set
do not run this
model.eval()
with torch.inference_mode():
    mlp_test_scaled = torch.cat([model(chunk) for chunk in x_test_t.split(1024)]).cpu().numpy()
mlp_pred = mlp_test_scaled.astype(np.float32) * scale + mean

# Retrieve linear model predictions
# If you used scikit-learn's LinearRegression:
linear_test_scaled = linear_model.predict(x_test.astype(np.float64))
linear_pred = linear_test_scaled.astype(np.float32) * scale + mean

# (If you used the PyTorch linear model, comment the two lines above and uncomment below)
# linear.eval()
# with torch.inference_mode():
#     linear_test_scaled = torch.cat([linear(chunk) for chunk in x_test_t.split(1024)]).cpu().numpy()
# linear_pred = linear_test_scaled.astype(np.float32) * scale + mean

# 2. Reconstruct test pairs and baseline predictions
test_pairs = pairs[test_idx]
# The additive baseline is Single A + Single B - WT
additive = y_single_ko[test_pairs[:, 0]] + y_single_ko[test_pairs[:, 1]] - y_wt
y_true = y_test

true_r = y_true - additive
add_pair_mse = np.mean(true_r ** 2, axis=1)
add_mse = float(np.mean(add_pair_mse))

# Helper function for Pearson correlation
def correlation(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    if np.std(a) < 1e-15 or np.std(b) < 1e-15: return None
    return float(np.corrcoef(a, b)[0, 1])

predictions = {"Additive Baseline": additive, "Linear Regression": linear_pred, "MLP": mlp_pred}
metrics, pair_losses = [], {}

# 3. Calculate core metrics
for name, pred in predictions.items():
    e2 = (pred - y_true) ** 2
    pair_losses[name] = e2.mean(axis=1)
    implied_residual = pred - additive
    
    mse = float(np.mean(e2))
    pearson_corr = correlation(implied_residual, true_r)
    
    metrics.append({
        "Model": name,
        "MSE": mse,
        "RMSE": mse ** 0.5,
        "MSE Improvement vs Additive (%)": 100 * (1 - mse / add_mse),
        "Pearson (Implied vs True Residual)": pearson_corr if pearson_corr else 0.0,
        "Fraction Pairs < Additive MSE": float(np.mean(pair_losses[name] < add_pair_mse))
    })

df = pd.DataFrame(metrics).set_index("Model")

# 4. Bootstrap 95% Confidence Intervals for MSE Improvement (2000 resamples)
rng = np.random.default_rng(9001)
boot = {"Linear Regression": [], "MLP": []}

for _ in range(2000):
    ii = rng.integers(0, len(test_pairs), size=len(test_pairs))
    denom = add_pair_mse[ii].mean()
    for name in boot:
        boot[name].append(100 * (1 - pair_losses[name][ii].mean() / denom))

# Calculate and format CIs
for name in boot:
    ci_lower, ci_upper = np.quantile(boot[name], [0.025, 0.975])
    df.loc[name, "MSE Improv. 95% CI"] = f"[{ci_lower:.2f}%, {ci_upper:.2f}%]"
    
df.loc["Additive Baseline", "MSE Improv. 95% CI"] = "-"

# 5. Calculate Pearson Correlation Improvement (MLP vs Linear)
mlp_pearson = df.loc["MLP", "Pearson (Implied vs True Residual)"]
lin_pearson = df.loc["Linear Regression", "Pearson (Implied vs True Residual)"]

if lin_pearson != 0:
    pearson_improvement = ((mlp_pearson - lin_pearson) / abs(lin_pearson)) * 100
    df.loc["MLP", "Pearson Improv. vs Linear (%)"] = pearson_improvement
df.loc["Linear Regression", "Pearson Improv. vs Linear (%)"] = 0.0

# Use np.nan instead of a string to keep the column numerically typed
df.loc["Additive Baseline", "Pearson Improv. vs Linear (%)"] = np.nan 

# Format the DataFrame for clean output
display_cols = ["RMSE", "MSE Improvement vs Additive (%)", "MSE Improv. 95% CI", 
                "Pearson (Implied vs True Residual)", "Pearson Improv. vs Linear (%)",
                "Fraction Pairs < Additive MSE"]

# Round the numbers, then fill the NaN values with "-" for a clean display
display(df[display_cols].round(4).fillna("-"))